In [ ]:
# 07 — Load synthetic operational data into Lakebase
# Copies the operational slice of the synthetic data into Lakebase (maintops schema):
#   * all users (100k clients + 10k handymen) and handyman_details
#   * incidents from the last ~3 months plus every still-active one (~45k rows)
# The full 1M-incident history stays in Delta only (README: never load it into Lakebase).
# Idempotent: runs in one transaction that first removes the previous synthetic load (ids in the
# synthetic ranges, and any incident that involves a synthetic user), then bulk-loads with COPY and
# moves the id sequences past the synthetic ranges so app-created rows never collide with them.

In [ ]:
%pip install -q "psycopg[binary]>=3.1"

In [ ]:
try:
    dbutils.library.restartPython()
except NameError:
    pass

In [ ]:
%run ./00_config

In [ ]:
# Cell 1 — What to load
import time
import psycopg
from pyspark.sql import functions as F

PG_URL       = dbutils.secrets.get("maintops", "lakebase_pg_url")
RECENT_FROM  = "2026-06-16"                 # ~3 months before HISTORY_END; older history stays in Delta
MAX_USER_ID  = N_USERS + N_HANDYMEN         # synthetic user ids are 1..MAX_USER_ID

USER_COLS = ["id", "email", "password_hash", "first_name", "last_name", "date_of_birth", "phone",
             "house", "postal_code", "city", "state", "country", "latitude", "longitude",
             "is_handyman", "created_at", "is_active"]
DETAIL_COLS = ["user_id", "specialisations", "skills", "experience_summary", "cv_path", "cv_raw_text",
               "has_car", "completed_cases", "rating_avg", "rating_count", "avg_price"]
INCIDENT_COLS = ["id", "reported_by_user_id", "handyman_user_id", "description", "incident_type", "urgency",
                 "recommended_handyman_ids", "agent_reasoning", "distance_km", "travel_time_minutes",
                 "status", "rating", "feedback", "created_at", "assigned_at", "completed_at", "updated_at"]

users     = spark.table(T_USERS)
details   = spark.table(T_HANDYMAN_DETAILS)
incidents = (spark.table(T_INCIDENTS)
             .where((F.col("created_at") >= F.lit(RECENT_FROM)) | F.col("status").isin("assigned", "in_progress")))
print(f"users={users.count():,}  handyman_details={details.count():,}  incidents={incidents.count():,}")

In [ ]:
# Cell 2 — Replace the synthetic slice in Lakebase (single transaction)
def copy_rows(cur, table, cols, df):
    """Stream a Spark DataFrame into a Lakebase table with COPY; returns the row count."""
    n = 0
    with cur.copy(f"COPY maintops.{table} ({', '.join(cols)}) FROM STDIN") as cp:
        for row in df.select(*cols).toLocalIterator():
            cp.write_row(tuple(row))
            n += 1
    return n

start = time.time()
with psycopg.connect(PG_URL, connect_timeout=30) as conn:
    with conn.cursor() as cur:
        cur.execute(
            "DELETE FROM maintops.incidents WHERE id <= %s OR reported_by_user_id <= %s OR handyman_user_id <= %s",
            (N_INCIDENTS, MAX_USER_ID, MAX_USER_ID),
        )
        cur.execute("DELETE FROM maintops.handyman_details WHERE user_id <= %s", (MAX_USER_ID,))
        cur.execute("DELETE FROM maintops.users WHERE id <= %s", (MAX_USER_ID,))

        loaded = {
            "users":            copy_rows(cur, "users", USER_COLS, users),
            "handyman_details": copy_rows(cur, "handyman_details", DETAIL_COLS, details),
            "incidents":        copy_rows(cur, "incidents", INCIDENT_COLS, incidents),
        }

        # App-created rows get ids above the synthetic ranges
        cur.execute("SELECT setval(pg_get_serial_sequence('maintops.users', 'id'), "
                    "GREATEST((SELECT max(id) FROM maintops.users), %s))", (MAX_USER_ID,))
        cur.execute("SELECT setval(pg_get_serial_sequence('maintops.incidents', 'id'), "
                    "GREATEST((SELECT max(id) FROM maintops.incidents), %s))", (N_INCIDENTS,))
    conn.commit()

    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute("ANALYZE maintops.users; ANALYZE maintops.handyman_details; ANALYZE maintops.incidents")

print(f"Loaded {loaded} in {time.time() - start:.0f}s")

In [ ]:
# Cell 3 — Check Lakebase matches what was loaded
with psycopg.connect(PG_URL, connect_timeout=30) as conn, conn.cursor() as cur:
    cur.execute("""
        SELECT (SELECT count(*) FROM maintops.users WHERE id <= %(u)s),
               (SELECT count(*) FROM maintops.handyman_details WHERE user_id <= %(u)s),
               (SELECT count(*) FROM maintops.incidents WHERE id <= %(i)s),
               (SELECT count(*) FROM maintops.incidents WHERE status IN ('assigned', 'in_progress'))
    """, {"u": MAX_USER_ID, "i": N_INCIDENTS})
    n_users, n_details, n_incidents, n_active = cur.fetchone()

print(f"Lakebase: users={n_users:,} handyman_details={n_details:,} incidents={n_incidents:,} (active {n_active:,})")
assert (n_users, n_details, n_incidents) == (loaded["users"], loaded["handyman_details"], loaded["incidents"]), \
    "Lakebase row counts differ from what was loaded"